<a href="https://colab.research.google.com/github/oumlk/french-wino-what/blob/main/corpus_construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### French Winograd Schemas dataset (Amsili & Seminck, 2017) XML




In [1]:
from google.colab import drive
drive.mount('/content/drive')

xml_path = "/content/drive/MyDrive/Thesis_2025/French_Wino_Schemas.xml"


Mounted at /content/drive


In [2]:
import xml.etree.ElementTree as ET

XML_PATH = "/content/drive/MyDrive/Thesis_2025/French_Wino_Schemas.xml"

tree = ET.parse(XML_PATH)
root = tree.getroot()

print("First 5 schemas from the XML file:")
for i, schema in enumerate(root.findall('.//schema')):
    if i >= 5:
        break
    print(f"Schema {i+1}:\n{ET.tostring(schema, encoding='unicode')}\n")

First 5 schemas from the XML file:
Schema 1:
<schema id="1" engn="02">
  <text>
    <txt1> La coupe n'entre pas dans la valise marron, car elle est trop </txt1>
    <wordA>grande</wordA>
    <wordB>petite</wordB>
    <txt2>.</txt2>
  </text>
  <question>
    <qn1>Qu'est-ce qui est trop </qn1>
    <qwordA>grand</qwordA>
    <qwordB>petit</qwordB>
    <qn2> ?</qn2>
  </question>
  <answer1>la coupe</answer1>
  <answer2>la valise</answer2>
</schema>



Schema 2:
<schema id="2" engn="04">
  <text>
    <txt1> Paul a essayé de joindre Georges sur son téléphone, mais il </txt1>
    <wordA>n'a pas réussi</wordA>
    <wordB>n'a pas répondu</wordB>
    <txt2>.</txt2>
  </text>
  <question>
    <qn1>Qui </qn1>
    <qwordA>n'a pas réussi</qwordA>
    <qwordB>n'a pas répondu</qwordB>
    <qn2> ?</qn2>
  </question>
  <answer1>Paul</answer1>
  <answer2>Georges</answer2>
</schema>



Schema 3:
<schema id="3" engn="05">
  <text>
    <txt1> L'avocat a posé une question au témoin, mais il a refusé </txt

In [4]:
import xml.etree.ElementTree as ET
import pandas as pd

XML_PATH = "/content/drive/MyDrive/Thesis_2025/French_Wino_Schemas.xml"
CSV_PATH = "/content/drive/MyDrive/Thesis_2025/french_winograd_schemas.csv"

tree = ET.parse(XML_PATH)
root = tree.getroot()

rows = []

for schema in root.findall(".//schema"):
    schema_id = schema.attrib.get("id")

    # Use findtext with a default value to safely extract text content
    # This prevents AttributeError if an element is missing
    text_element = schema.find("text")
    if text_element is not None:
        txt1 = text_element.findtext("txt1", default="").strip()
        txt2 = text_element.findtext("txt2", default="").strip()
        wordA = text_element.findtext("wordA", default="").strip()
        wordB = text_element.findtext("wordB", default="").strip()
        sentence = f"{txt1} _ {txt2}"
    else:
        txt1 = ""
        txt2 = ""
        wordA = ""
        wordB = ""
        sentence = ""

    answer1 = schema.findtext("answer1", default="").strip()
    answer2 = schema.findtext("answer2", default="").strip()

    rows.append({
        "id": schema_id,
        "sentence": sentence,
        "option1": wordA,
        "option2": wordB,
        "answer1": answer1,
        "answer2": answer2
    })

df = pd.DataFrame(rows)

df.head()

,id,sentence,option1,option2,answer1,answer2
0,1,"La coupe n'entre pas dans la valise marron, ca...",grande,petite,la coupe,la valise
1,2,Paul a essayé de joindre Georges sur son télép...,n'a pas réussi,n'a pas répondu,Paul,Georges
2,3,"L'avocat a posé une question au témoin, mais i...",de la répéter,d'y répondre,l'avocat,le témoin
3,4,Nicolas n'a pas pu soulever son fils car il ét...,faible,lourd,Nicolas,son fils
4,5,"Les lycéens harcelaient les collégiens, donc o...",punis,défendus,les lycéens,les collégiens


In [5]:
df.to_csv(CSV_PATH, index=False)

print(f"Saved {len(df)} schemas to {CSV_PATH}")

Saved 107 schemas to /content/drive/MyDrive/Thesis_2025/french_winograd_schemas.csv


After converting the original XML file into a tabular format, I performed an intermediate normalization step to make the structure compatible with the WinoWhat framework. In this step, the placeholder "_" in each sentence is temporarily filled with the first candidate option1, and the original option columns are removed. The two candidate antecedents provided in the dataset are then stored as option1 and option2, corresponding to the entities implied by each alternative completion. A new column answer is added to encode the correct choice as a binary label. In a subsequent step, the placeholder token _ is reintroduced into the sentence to match the surface format used in the English WinoWhat dataset.

In [6]:
import pandas as pd

# 1. Replace "_" in the sentence with the original option1
df["sentence"] = df.apply(
    lambda row: row["sentence"].replace("_", row["option1"], 1),
    axis=1
)

# 2. Drop the original option1 and option2 columns (the fillers)
df = df.drop(columns=["option1", "option2"])

# 3. Rename answer columns to option columns (entities)
df = df.rename(columns={
    "answer1": "option1",
    "answer2": "option2"
})

# 4. Add an empty answer column (to be filled manually)
df["answer"] = ""

# Inspect result
df.head()


,id,sentence,option1,option2,answer
0,1,"La coupe n'entre pas dans la valise marron, ca...",la coupe,la valise,
1,2,Paul a essayé de joindre Georges sur son télép...,Paul,Georges,
2,3,"L'avocat a posé une question au témoin, mais i...",l'avocat,le témoin,
3,4,Nicolas n'a pas pu soulever son fils car il ét...,Nicolas,son fils,
4,5,"Les lycéens harcelaient les collégiens, donc o...",les lycéens,les collégiens,


In [7]:
# Save to CSV
OUTPUT_CSV_PATH = "/content/drive/MyDrive/Thesis_2025/french_winograd_intermediate_entities.csv"
df.to_csv(OUTPUT_CSV_PATH, index=False)

print(f"Saved transformed dataset to: {OUTPUT_CSV_PATH}")


Saved transformed dataset to: /content/drive/MyDrive/Thesis_2025/french_winograd_intermediate_entities.csv


The initial version of the dataset (french_wino_unrandomized.csv) was constructed such that the correct answer was consistently stored as the first candidate option. While this representation preserves the intended commonsense interpretation of each instance, it may introduce positional bias during model evaluation.

To mitigate this effect, we perform an option randomization step. For each instance independently, the order of the two candidate options is randomly swapped with a probability of 0.5. When a swap occurs, the gold label is updated accordingly (i.e., labels 1 and 2 are exchanged). This procedure preserves the semantic correctness of each instance while ensuring that the position of the correct answer is not systematically correlated with a fixed option index.

The randomized dataset is stored as french_wino_randomized.csv. A fixed random seed is used to ensure reproducibility

In [8]:
import pandas as pd
import numpy as np

# Paths
INPUT_CSV = "/content/drive/MyDrive/Thesis_2025/french_wino_unrandomized.csv"
OUTPUT_CSV = "/content/drive/MyDrive/Thesis_2025/french_wino_randomized.csv"

# Load data
df = pd.read_csv(INPUT_CSV)

# Set seed for reproducibility
np.random.seed(42)

def randomize_row(row):
    # 50% chance to swap
    if np.random.rand() < 0.5:
        # Swap options
        row["option1"], row["option2"] = row["option2"], row["option1"]

        # Flip answer
        if row["answer"] == 1:
            row["answer"] = 2
        elif row["answer"] == 2:
            row["answer"] = 1

    return row

# Apply randomization
df = df.apply(randomize_row, axis=1)

# Save randomized dataset
df.to_csv(OUTPUT_CSV, index=False)

print(f"Randomized dataset saved to: {OUTPUT_CSV}")
df.head()


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Thesis_2025/french_wino_unrandomized.csv'

In [9]:
# Add paraphrased column:
if "paraphrased_sentence" not in df.columns:
    df["paraphrased_sentence"] = ""

df.to_csv(OUTPUT_CSV, index=False)

df.head()

,id,sentence,option1,option2,answer,paraphrased_sentence
0,1,"La coupe n'entre pas dans la valise marron, ca...",la coupe,la valise,,
1,2,Paul a essayé de joindre Georges sur son télép...,Paul,Georges,,
2,3,"L'avocat a posé une question au témoin, mais i...",l'avocat,le témoin,,
3,4,Nicolas n'a pas pu soulever son fils car il ét...,Nicolas,son fils,,
4,5,"Les lycéens harcelaient les collégiens, donc o...",les lycéens,les collégiens,,


In [10]:
import pandas as pd

# my final paraphrased file
df = pd.read_csv('/content/drive/MyDrive/Thesis_2025/french_wino_with_paraphrases.csv')

# added source column based on ID split
# IDs 1-97 = Amsili & Seminck, IDs 98+ = WinoWhat
df['source'] = df['id'].apply(lambda x: 'amsili_seminck' if x <= 97 else 'winowhat')

# check the split
print(df['source'].value_counts())
print()
print("First 5 rows:")
print(df[['id', 'source', 'sentence']].head())
print()
print("Around the split point (IDs 95-100):")
print(df[df['id'].between(95, 101)][['id', 'source', 'sentence']].to_string())

# save
df.to_csv('french_wino_final_with_source.csv', index=False)
print("\nSaved to french_wino_final_with_source.csv")

source
winowhat          104
amsili_seminck     96
Name: count, dtype: int64

First 5 rows:
   id          source                                           sentence
0   1  amsili_seminck  La coupe n'entre pas dans la valise marron, ca...
1   2  amsili_seminck  Paul a essayé de joindre Georges sur son télép...
2   3  amsili_seminck  L'avocat a posé une question au témoin, mais _...
3   4  amsili_seminck  Nicolas n'a pas pu soulever son fils car _ éta...
4   5  amsili_seminck  Les lycéens harcelaient les collégiens, donc o...

Around the split point (IDs 95-100):
     id          source                                                                                                                                                                                sentence
93   95  amsili_seminck                                                                    Samuel et Amélie sont passionnément amoureux mais les parents d'Amélie sont contre cette relation car _ sont jeunes.
94   96  amsili_

In [11]:
output_path = '/content/drive/MyDrive/Thesis_2025/french_wino_final_with_source.csv'

df.to_csv(output_path, index=False)

print(f"\nSaved to {output_path}")


Saved to /content/drive/MyDrive/Thesis_2025/french_wino_final_with_source.csv


In [12]:
import pandas as pd
df = pd.read_csv('french_wino_final_with_source.csv')

# display the 4 rows that needed manual manipulation
print(df[df['id'].isin([28, 34, 41, 46])][
    ['id','sentence','option1','option2','answer']
].to_string())

    id                                                                                                                                                      sentence                 option1                  option2  answer
27  28             Il y a trop de cerfs dans le parc, et les gardes forestiers ont introduit une petite meute de loups. La population _ devrait rapidement diminuer.               des cerfs                des loups       1
33  34  Tout le monde a aimé les biscuits à l'avoine; seules quelques personnes ont préféré ceux aux pépites de chocolat. La prochaine fois, il faudra faire plus _.  de biscuits à l'avoine  de biscuits au chocolat       1
40  41                                                                                                           Gérard a appelé le barman. _ a rendu sa pinte vide.               Le barman                   Gérard       2
45  46                                                                   Le Dr. Vincenot a informé Michel qu'il 

In [15]:
import pandas as pd


CSV_PATH = "/content/drive/MyDrive/Thesis_2025/french_wino_with_paraphrases.csv"

df = pd.read_csv(CSV_PATH)

# strip whitespace and check if sentence ends indeed with _
# this handles endings like "_ ." or "_." or "_ " as well as bare "_"
def ends_with_blank(sentence):
    if not isinstance(sentence, str):
        return False

    stripped = sentence.strip().rstrip('.!?,;:').strip()
    return stripped.endswith('_')

df['ends_with_blank'] = df['paraphrased_sentence'].apply(ends_with_blank)

total = len(df)
ends_blank = df['ends_with_blank'].sum()
not_ends_blank = total - ends_blank

print(f"Total instances:              {total}")
print(f"Paraphrases ending with _:    {ends_blank}  ({ends_blank/total*100:.1f}%)")
print(f"Paraphrases NOT ending with _:{not_ends_blank}  ({not_ends_blank/total*100:.1f}%)")

print()
print("Sample of sentences NOT ending with blank")
print(df[~df['ends_with_blank']][['id', 'paraphrased_sentence']].head(10).to_string())

Total instances:              200
Paraphrases ending with _:    96  (48.0%)
Paraphrases NOT ending with _:104  (52.0%)

Sample of sentences NOT ending with blank
    id                                                                                                         paraphrased_sentence
0    1                                         Impossible de faire entrer la coupe dans la valise marron puisque _ est trop grande.
1    2                                    Paul a tenté d'appeler Georges sur son portable, cependant _ n’a pas réussi à le joindre.
2    3             L'avocat a posé une question au témoin ; ce dernier a essuyé un refus de _, qui a décliné de répéter la question
3    4                                                         Comme _ était trop faible, Nicolas n’a pas réussi à porter son fils.
5    6                                         L’eau de la carafe a été transvasée dans la tasse jusqu’à ce que _ se retrouve vide.
6    7                           Les frais de 